In [ ]:
""" 均线交叉策略和信号延迟
SMA：常见的选择有50天和200天，
EMA：常见的有10天和30天，或者在高频的情况下，为5分钟和5分钟
well，和所有的金融指标类似，在使用的使用，要weekly和daily一块用（参照同时间的指标），并且要和其他金融指标参照使用，

金融领域中常见的bias有：lookahead bias，survivorship bias，psychology- tolerance bias
几乎所有产品或计算都可能会遇到上面的偏差

.astype(int),表示转换为什么类型a
axis = 1 依然是对每一行中的列，进行操作，注意分清楚.mean(axis = 1)和.sum(axis=1)
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ema5 = df_close.ewm(span = 5,adjust = False).mean()
ema20 = df_close.ewm(span = 20,adjust = False).mean()
log_returns = np.log(df_close/df_close.shift(1))
conditions = [
    ema5>ema20,
    ema5<ema20,
    (ema5<=ema20)&(ema5>=ema20)
]

choices = [1,-1,0]

signal_ema = pd.DataFrame(
    np.select(conditions,choices,default = 0),
    index = df_close.index,
    columns = df_close.columns
)

signal_ema_delayed = signal_ema.shift(1)

strategy_log_returns = signal_ema_delayed*log_returns
strategy_log_returns_nodelayed = signal_ema*log_returns

cum_ema = strategy_log_returns.sum(axis = 1).cumsum().apply(np.exp)
cum_ema_nodelayed = strategy_log_returns_nodelayed.sum(axis = 1).cumsum().apply(np.exp)

fig,ax = plt.subplots(figsize = (12,6))

ax.plot(cum_ema,label = 'ema cross strategy with delayed',alpha = 0.7)
ax.plot(cum_ema_nodelayed, label = 'ema cross strategy without delayed',alpha = 0.7)
ax.legend()
ax.set_title('the delayed ema VS the No delayed ema')


In [ ]:
def annual_return_cc(cum_series):
    total_log_return = cum_series.iloc[-1]
    n_years = len(cum_series)/252
    return np.exp(total_log_return/n_years)-1
print(f"无延迟年化收益: {annual_return_cc(cum_ema_nodelayed):.2%}")
print(f"有延迟年化收益: {annual_return_cc(cum_ema):.2%}")

In [ ]:
signal_changes = (signal_ema.diff().abs()>0).astype(int)
turnover = signal_changes / signal_changes.shape[1]
#shape[1]表示的列数
avg_turnover = turnover.mean()
print(f"日均换手率: {avg_turnover:.2%}")

In [ ]:
commission_rate = 0.0015
trades = signal_ema.diff().abs()
commission_cost = trades * commission_rate
strategy_log_returns_net = strategy_log_returns - commission_cost
cum_net = strategy_log_returns_net.sum(axis=1).cumsum().apply(np.exp)

print(f"扣除手续费前年化收益: {annual_return_cc(cum_ema):.2%}")
print(f"扣除手续费后年化收益: {annual_return_cc(cum_net):.2%}")